In [ ]:
# !git clone https://github.com/alipay/Spatio-Temporal-Hypergraph-Model.git
# %cd Spatio-Temporal-Hypergraph-Model
# !pip install -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount('/absolute/path/to')

In [ ]:
# from drive to colab
!cp -r "/absolute/path/to/Spatio-Temporal-Hypergraph-Model" \
      /absolute/path/to/Spatio-Temporal-Hypergraph-Model

In [ ]:
%cd Spatio-Temporal-Hypergraph-Model

In [ ]:
import pickle as pkl
from pathlib import Path
import pandas as pd
import os
import glob

In [ ]:
!pip install "numpy<2"

In [ ]:
import numpy as np
np.__version__

Restart session, import again and continue...

In [ ]:
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric \
  -f https://data.pyg.org/whl/torch-2.6.0+cu124.html

import torch
print(torch.__version__, torch.version.cuda)
import torch_scatter, torch_sparse, torch_cluster, torch_spline_conv, torch_geometric
print("✔️ PyG imports OK")

In [ ]:
original_path = '/absolute/path/to/SafetyIsAllYouNeed/baselines/exports'
base_path =  '/absolute/path/to/Spatio-Temporal-Hypergraph-Model'

### Load original trajectories

In [ ]:
with open(os.path.join(original_path, 'train_trajectories.pickle'), 'rb') as f:
    train_trajs = pkl.load(f)

with open(os.path.join(original_path, 'validation_trajectories.pickle'), 'rb') as f:
    validation_trajs = pkl.load(f)

with open(os.path.join(original_path, 'test_trajectories.pickle'), 'rb') as f:
    test_trajs = pkl.load(f)

train_df = pd.read_csv(os.path.join(original_path, 'train_checkins.csv'))
validation_df = pd.read_csv(os.path.join(original_path, 'validation_checkins.csv'))
test_df = pd.read_csv(os.path.join(original_path, 'test_checkins.csv'))

In [ ]:
all_cats = pd.concat(train_trajs + validation_trajs + test_trajs)["poi_category_id"].unique()
cat2code = {cat: i for i,cat in enumerate(sorted(all_cats))}

def dump_sthg(trajectories, out_path):
    rows = []
    for traj in trajectories:
        # traj is a DF of 20 check-ins sorted by local_time
        uid = traj.user_id.iloc[0]
        # use the existing trajectory_id if you set one, else build your own:
        if "trajectory_id" in traj:
            wid = traj.trajectory_id.iloc[0]
        else:
            # fallback: useridx_position
            wid = f"{uid}_{traj.index.min()}"
        first_day = traj.local_time.dt.normalize().iloc[0]

        # now row by row
        for _, row in traj.iterrows():
            lt = row.local_time
            dow = lt.weekday()
            secs = lt.hour*3600 + lt.minute*60 + lt.second
            norm_in_day = secs / (24*3600)
            day_shift = (lt.normalize() - first_day).days
            rows.append({
                "user_id":           uid,
                "POI_id":            row.poi_id,
                "POI_catid":         row.poi_category_id,
                "POI_catid_code":    cat2code[row.poi_category_id],
                "POI_catname":       row.poi_category_name,
                "latitude":          row.latitude,
                "longitude":         row.longitude,
                "timezone":          row.timezone_offset,
                "UTC_time":          row.utc_time,
                "local_time":        lt,
                "day_of_week":       dow,
                "norm_in_day_time":  norm_in_day,
                "trajectory_id":     wid,
                "norm_day_shift":    day_shift,
                # we'll fill norm_relative_time afterwards per‐trajectory
                "_pos_in_traj":      row.name  # temporary, use the DF index or trajectory offset
            })
    df = pd.DataFrame(rows)

    # compute norm_relative_time: within each trajectory 0.0→1.0
    df["pos"] = df.groupby("trajectory_id").cumcount()
    traj_lens = df.groupby("trajectory_id")["pos"].transform("max")
    df["norm_relative_time"] = df["pos"] / traj_lens.astype(float)

    # select & reorder to exactly their 15 columns
    out = df[[
      "user_id","POI_id","POI_catid","POI_catid_code","POI_catname",
      "latitude","longitude","timezone","UTC_time","local_time",
      "day_of_week","norm_in_day_time","trajectory_id",
      "norm_day_shift","norm_relative_time"
    ]]

    # ensure output directory exists
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    # write with no header, tab‐separated
    out.to_csv(out_path, sep="\t", header=True, index=False)

In [ ]:
dump_sthg(train_trajs, os.path.join(base_path, "data/CHICAGO/CHICAGO_train.tsv"))
dump_sthg(validation_trajs, os.path.join(base_path, "data/CHICAGO/CHICAGO_val.tsv"))
dump_sthg(test_trajs, os.path.join(base_path, "data/CHICAGO/CHICAGO_test.tsv"))
# !cp "/absolute/path/to/Spatio-Temporal-Hypergraph-Model/data/CHICAGO/CHICAGO_train.tsv"  data/CHICAGO/CHICAGO_train.tsv
# !cp "/absolute/path/to/Spatio-Temporal-Hypergraph-Model/data/CHICAGO/CHICAGO_val.tsv"    data/CHICAGO/CHICAGO_val.tsv
# !cp "/absolute/path/to/Spatio-Temporal-Hypergraph-Model/data/CHICAGO/CHICAGO_test.tsv"   data/CHICAGO/CHICAGO_test.tsv

In [ ]:
# repo_root = Path("/absolute/path/to/Spatio-Temporal-Hypergraph-Model")
# src_dir = repo_root / "data" / "nyc" / "raw"
# raw_dir = repo_root / "data" / "nyc" / "raw"
# for split in ("train", "val", "test"):
#     tsv = src_dir / f"CHICAGO_{split}.tsv"
#     assert tsv.exists(), f"Missing {tsv}"
#     df = pd.read_csv(tsv, sep="\t", header=0)    # header=0 because you saved with header=True
#     out_csv = raw_dir / f"CHICAGO_{split}.csv"
#     print(f"→ {tsv.name} → {out_csv}")
#     df.to_csv(out_csv, index=False)

## We want every trajectory to be exactly 20 steps

In [ ]:
RAW_DIR = "data/nyc/raw"
OUT_DIR = "data/nyc/raw_len20"
os.makedirs(OUT_DIR, exist_ok=True)

for split in ["train", "val", "test"]:
    df  = pd.read_csv(f"{RAW_DIR}/CHICAGO_{split}.csv")

    # trajectories that have at least 20 records
    long = (df.groupby("trajectory_id")
              .filter(lambda g: len(g) >= 20))

    # sort chronologically *within traj* and keep the **last** 20
    long = (long.sort_values("local_time")
                 .groupby("trajectory_id")
                 .tail(20))

    long.to_csv(f"{OUT_DIR}/CHICAGO_{split}.csv", index=False)

In [ ]:
import torch, torch_geometric
print(torch.__version__, torch.version.cuda, torch.cuda.is_available())

In [ ]:
# # Clean slate
# !pip uninstall -y torch torchvision torchaudio torchtext torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric

# # Reinstall torch with CUDA (cu117), compatible with PyG wheels
# !pip install --no-cache-dir torch==2.0.1+cu117 torchvision==0.15.2+cu117 torchaudio==2.0.2+cu117 \
#   -f https://download.pytorch.org/whl/torch_stable.html

# # Now install PyG and its extensions compatible with torch 2.0.1 + cu117
# !pip install --no-cache-dir torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric \
#   -f https://data.pyg.org/whl/torch-2.0.1+cu117.html


In [ ]:
!python run.py -f best_conf/nyc.yml

_(Removed stale Colab error output; see upstream baseline repo logs if needed.)_

Copy from colab to drive

In [ ]:
!cp -r /absolute/path/to/Spatio-Temporal-Hypergraph-Model \
       "/absolute/path/to/Spatio-Temporal-Hypergraph-Model_2"

In [ ]:
!grep -R "Test evaluation result" log/*/nyc/train.log